# Ch07 — 假設檢定框架 (Hypothesis Testing Framework)

**預計時間：** 2 小時

從概念到實際檢定的橋樑 — 本章建立完整的假設檢定思維框架，為後續各種統計檢定奠定基礎。

## 學習目標

完成本章後，你將能夠：

1. 正確陳述虛無假設 (Null Hypothesis) 與對立假設 (Alternative Hypothesis)
2. 解釋 p-value 的真正意義，並根據顯著水準做出決策
3. 區分單尾檢定 (One-tailed Test) 與雙尾檢定 (Two-tailed Test) 的適用情境
4. 辨別 Type I Error 與 Type II Error，理解兩者的取捨關係
5. 運用統計檢定力 (Power) 與效果量 (Effect Size) 設計合理的實驗
6. 破除對 p-value 的常見迷思

### 前置章節

- [Ch06 — 點估計與信賴區間](06_點估計與信賴區間.ipynb)：抽樣分佈、標準誤、信賴區間的概念

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from statsmodels.stats.power import tt_solve_power, tt_ind_solve_power

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 圖表風格
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

%matplotlib inline

In [ ]:
# 載入 Titanic 資料集
titanic = pd.read_csv('datasets/titanic_train.csv')
print(f'資料筆數: {titanic.shape[0]}, 欄位數: {titanic.shape[1]}')
titanic.head(3)

In [ ]:
# 快速檢視年齡欄位的基本統計量
print("=== Titanic 年齡欄位概覽 ===")
print(f"非遺漏值筆數: {titanic['Age'].notna().sum()}")
print(f"遺漏值筆數:   {titanic['Age'].isna().sum()}")
print()
titanic['Age'].describe()

---
## Part A — 假設檢定的邏輯 (Logic of Hypothesis Testing)

### A1. 虛無假設 H$_0$ vs 對立假設 H$_1$

假設檢定是一套**反證法**的推論框架：

| 概念 | 說明 | 範例 |
|------|------|------|
| **虛無假設 H$_0$** (Null Hypothesis) | 「沒有差異」或「沒有效果」的預設立場 | Titanic 男女乘客平均年齡相同 |
| **對立假設 H$_1$** (Alternative Hypothesis) | 研究者想要證明的主張 | Titanic 男女乘客平均年齡不同 |

> **核心邏輯：** 我們不直接「證明」H$_1$ 為真，而是嘗試在資料中找到「足夠強的證據」來**拒絕** H$_0$。

### A2. 檢定統計量 (Test Statistic)

檢定統計量是一個**數值摘要**，用來量化「觀察到的資料與 H$_0$ 預期之間的偏離程度」。

常見的檢定統計量：

- **z 統計量**：已知母體標準差時使用
- **t 統計量**：未知母體標準差、使用樣本標準差估計時
- **$\chi^2$ 統計量**：用於類別變數的檢定
- **F 統計量**：用於多組比較（ANOVA）

$$
\text{檢定統計量} = \frac{\text{觀察值} - \text{H}_0 \text{期望值}}{\text{標準誤}}
$$

In [ ]:
# 示範：手動計算 t 統計量
titanic_age = titanic["Age"].dropna()
n = len(titanic_age)
x_bar = titanic_age.mean()
mu_0 = 30  # H0 假設的母體均值
se = titanic_age.std(ddof=1) / np.sqrt(n)

t_manual = (x_bar - mu_0) / se

print(f"樣本均值 x-bar = {x_bar:.4f}")
print(f"H0 假設均值 mu_0 = {mu_0}")
print(f"標準誤 SE = {se:.4f}")
print(f"手動計算 t = (x-bar - mu_0) / SE = {t_manual:.4f}")

# 驗證：與 scipy 結果一致
t_scipy, p_scipy = stats.ttest_1samp(titanic_age, popmean=mu_0)
print(f"scipy 計算 t = {t_scipy:.4f}")
print(f"\n兩者一致！")

### A3. p-value 的定義

> **p-value（p 值）的嚴謹定義：**
> 
> **假設 H$_0$ 為真的前提下**，觀察到與目前資料一樣極端（或更極端）結果的機率。

p-value **不是** H$_0$ 為真的機率！它是一個條件機率：

$$
p\text{-value} = P(\text{資料這麼極端} \mid H_0 \text{ 為真})
$$

In [ ]:
# 視覺化 p-value 的概念
# 情境：H0 為 mu = 30，觀察到的樣本均值為 32

fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(-4, 4, 500)
y = stats.norm.pdf(x)

# 假設 H0 下的分佈
ax.plot(x, y, 'k-', lw=2, label='H$_0$ 下的抽樣分佈')
ax.fill_between(x, y, where=(x >= 2), color='red', alpha=0.4, label='p-value（右尾）')
ax.fill_between(x, y, where=(x <= -2), color='red', alpha=0.4, label='p-value（左尾）')

ax.axvline(x=2, color='red', linestyle='--', lw=1.5)
ax.axvline(x=-2, color='red', linestyle='--', lw=1.5)

ax.set_xlabel('檢定統計量', fontsize=12)
ax.set_ylabel('機率密度', fontsize=12)
ax.set_title('p-value 示意圖（雙尾檢定）', fontsize=14)
ax.legend(fontsize=11)

# 標註
ax.annotate('觀察到的\n檢定統計量', xy=(2, 0.01), xytext=(3, 0.15),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='red'),
            color='red')

p_val = 2 * stats.norm.sf(2)
ax.text(0, 0.25, f'紅色面積 = p-value = {p_val:.4f}',
        fontsize=12, ha='center', color='red',
        bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()

### A4. 顯著水準 $\alpha$ 與決策規則

**顯著水準 $\alpha$**（Significance Level）是事先設定的「門檻值」，常見值為 0.05、0.01、0.10。

| 決策規則 | 條件 | 結論 |
|----------|------|------|
| **拒絕 H$_0$** | p-value < $\alpha$ | 有統計顯著的證據支持 H$_1$ |
| **不拒絕 H$_0$** | p-value $\geq$ $\alpha$ | 沒有足夠證據拒絕 H$_0$ |

> **口訣：「p 小拒 H$_0$」** — p-value 越小，拒絕 H$_0$ 的證據越強。

注意：「不拒絕 H$_0$」**不等於**「接受 H$_0$」。我們只是說「目前的證據不足以推翻 H$_0$」。

In [ ]:
# 實際範例：Titanic 乘客的平均年齡是否等於 30 歲？
# H0: mu = 30
# H1: mu != 30

titanic_age = titanic['Age'].dropna()

t_stat, p_value = stats.ttest_1samp(titanic_age, popmean=30)

alpha = 0.05
print(f'樣本均值: {titanic_age.mean():.2f}')
print(f'樣本大小: {len(titanic_age)}')
print(f't 統計量: {t_stat:.4f}')
print(f'p-value:  {p_value:.4f}')
print(f'顯著水準: {alpha}')
print(f'\n決策: {"拒絕 H0" if p_value < alpha else "不拒絕 H0"}')
print(f'結論: {"有足夠證據顯示 Titanic 乘客平均年齡不等於 30 歲" if p_value < alpha else "沒有足夠證據顯示平均年齡不等於 30 歲"}')

---
## Part B — 單尾 vs 雙尾檢定 (One-tailed vs Two-tailed Tests)

In [ ]:
# 視覺化：不同 alpha 值的拒絕域

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.linspace(-4, 4, 500)
y = stats.norm.pdf(x)

for ax, a, title in zip(axes,
                        [0.10, 0.05, 0.01],
                        ["alpha = 0.10", "alpha = 0.05", "alpha = 0.01"]):
    z_c = stats.norm.ppf(1 - a/2)
    ax.plot(x, y, "k-", lw=2)
    ax.fill_between(x, y, where=(x <= -z_c), color="red", alpha=0.4)
    ax.fill_between(x, y, where=(x >= z_c), color="red", alpha=0.4)
    ax.fill_between(x, y, where=((x > -z_c) & (x < z_c)), color="grey", alpha=0.15)
    ax.axvline(-z_c, color="red", ls="--", lw=1)
    ax.axvline(z_c, color="red", ls="--", lw=1)
    ax.set_title(title, fontsize=13)
    ax.text(0, 0.2, f"不拒絕域\n{(1-a)*100:.0f}%", ha="center", fontsize=10)

plt.suptitle("alpha 越小 -> 拒絕域（紅色）越窄 -> 越難拒絕 H0", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

### A5. 假設檢定的標準流程



### B1. 概念比較

| 檢定類型 | H$_1$ 方向 | 拒絕域位置 | 適用情境 |
|----------|-----------|-----------|----------|
| **雙尾檢定** (Two-tailed) | $\mu \neq \mu_0$ | 兩側各 $\alpha/2$ | 不確定方向時 |
| **右尾檢定** (Right-tailed) | $\mu > \mu_0$ | 右側 $\alpha$ | 預期「增加」時 |
| **左尾檢定** (Left-tailed) | $\mu < \mu_0$ | 左側 $\alpha$ | 預期「減少」時 |

In [ ]:
# 圖解：單尾 vs 雙尾的拒絕域

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
x = np.linspace(-4, 4, 500)
y = stats.norm.pdf(x)
alpha = 0.05

# --- 左尾檢定 ---
ax = axes[0]
z_left = stats.norm.ppf(alpha)
ax.plot(x, y, 'k-', lw=2)
ax.fill_between(x, y, where=(x <= z_left), color='red', alpha=0.5, label=f'拒絕域 (alpha={alpha})')
ax.fill_between(x, y, where=(x > z_left), color='grey', alpha=0.2, label='不拒絕域')
ax.axvline(z_left, color='red', ls='--', lw=1.5)
ax.set_title('左尾檢定 (Left-tailed)', fontsize=13)
ax.text(z_left - 0.3, 0.05, f'z={z_left:.2f}', color='red', fontsize=10, ha='right')
ax.legend(fontsize=9)

# --- 右尾檢定 ---
ax = axes[1]
z_right = stats.norm.ppf(1 - alpha)
ax.plot(x, y, 'k-', lw=2)
ax.fill_between(x, y, where=(x >= z_right), color='red', alpha=0.5, label=f'拒絕域 (alpha={alpha})')
ax.fill_between(x, y, where=(x < z_right), color='grey', alpha=0.2, label='不拒絕域')
ax.axvline(z_right, color='red', ls='--', lw=1.5)
ax.set_title('右尾檢定 (Right-tailed)', fontsize=13)
ax.text(z_right + 0.1, 0.05, f'z={z_right:.2f}', color='red', fontsize=10)
ax.legend(fontsize=9)

# --- 雙尾檢定 ---
ax = axes[2]
z_lo = stats.norm.ppf(alpha / 2)
z_hi = stats.norm.ppf(1 - alpha / 2)
ax.plot(x, y, 'k-', lw=2)
ax.fill_between(x, y, where=(x <= z_lo), color='red', alpha=0.5, label=f'拒絕域 (各 alpha/2)')
ax.fill_between(x, y, where=(x >= z_hi), color='red', alpha=0.5)
ax.fill_between(x, y, where=((x > z_lo) & (x < z_hi)), color='grey', alpha=0.2, label='不拒絕域')
ax.axvline(z_lo, color='red', ls='--', lw=1.5)
ax.axvline(z_hi, color='red', ls='--', lw=1.5)
ax.set_title('雙尾檢定 (Two-tailed)', fontsize=13)
ax.text(z_lo - 0.3, 0.05, f'z={z_lo:.2f}', color='red', fontsize=10, ha='right')
ax.text(z_hi + 0.1, 0.05, f'z={z_hi:.2f}', color='red', fontsize=10)
ax.legend(fontsize=9)

for ax in axes:
    ax.set_xlabel('z 值', fontsize=11)
    ax.set_ylabel('機率密度', fontsize=11)

plt.suptitle('三種假設檢定的拒絕域比較', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# 單尾 vs 雙尾的 p-value 比較
# 情境：Titanic 乘客平均年齡是否「大於」28 歲？

titanic_age = titanic['Age'].dropna()
t_stat, p_two = stats.ttest_1samp(titanic_age, popmean=28)

# 右尾 p-value
p_right = p_two / 2 if t_stat > 0 else 1 - p_two / 2

print('=== 雙尾檢定 ===')
print(f'H0: mu = 28,  H1: mu != 28')
print(f't = {t_stat:.4f},  p-value = {p_two:.4f}')
print(f'決策 (alpha=0.05): {"拒絕 H0" if p_two < 0.05 else "不拒絕 H0"}')

print(f'\n=== 右尾檢定 ===')
print(f'H0: mu <= 28,  H1: mu > 28')
print(f't = {t_stat:.4f},  p-value = {p_right:.4f}')
print(f'決策 (alpha=0.05): {"拒絕 H0" if p_right < 0.05 else "不拒絕 H0"}')

### B2. 如何選擇？

> **原則：** 在收集資料**之前**就決定好用單尾還是雙尾。

- 如果你的研究假設有**明確方向**（例如「新藥比舊藥有效」），使用**單尾檢定**
- 如果你只是想知道「兩者是否不同」，使用**雙尾檢定**
- 不確定時，**預設使用雙尾檢定**（比較保守）

> **注意：** 不可以先看資料方向再決定用單尾 — 這會膨脹 Type I Error！

---
## Part C — Type I / Type II Error（型一/型二錯誤）

### C1. 兩種錯誤的定義

| | H$_0$ 實際為**真** | H$_0$ 實際為**假** |
|---|---|---|
| **不拒絕 H$_0$** | 正確決策 (1 - $\alpha$) | **Type II Error ($\beta$)** 偽陰性 |
| **拒絕 H$_0$** | **Type I Error ($\alpha$)** 偽陽性 | 正確決策 (Power = 1 - $\beta$) |

- **Type I Error（型一錯誤，$\alpha$）**：H$_0$ 為真卻拒絕了它 → **偽陽性 (False Positive)**
  - 類比：健康的人被誤診為生病
  
- **Type II Error（型二錯誤，$\beta$）**：H$_0$ 為假卻不拒絕它 → **偽陰性 (False Negative)**
  - 類比：生病的人被誤診為健康

In [ ]:
# Type I / Type II Error 視覺化
# 重用舊版圖表邏輯，翻譯為繁體中文

fig, ax = plt.subplots(figsize=(12, 7))

x_range = np.arange(-4, 10, 0.01)

# === 虛無假設 H0 分佈 (N(0,1)) ===
# 左尾拒絕域 (Type I Error)
ax.fill_between(x=np.arange(-4, -2, 0.01),
                y1=stats.norm.pdf(np.arange(-4, -2, 0.01)),
                facecolor='red', alpha=0.35)

# H0 不拒絕域
ax.fill_between(x=np.arange(-2, 2, 0.01),
                y1=stats.norm.pdf(np.arange(-2, 2, 0.01)),
                facecolor='grey', alpha=0.35)

# 右尾拒絕域 (Type I Error)
ax.fill_between(x=np.arange(2, 4, 0.01),
                y1=stats.norm.pdf(np.arange(2, 4, 0.01)),
                facecolor='red', alpha=0.5)

# === 對立假設 H1 分佈 (N(3,2)) ===
# H1 在 H0 不拒絕域內的部分 (Type II Error)
ax.fill_between(x=np.arange(-4, -2, 0.01),
                y1=stats.norm.pdf(np.arange(-4, -2, 0.01), loc=3, scale=2),
                facecolor='grey', alpha=0.35)

ax.fill_between(x=np.arange(-2, 2, 0.01),
                y1=stats.norm.pdf(np.arange(-2, 2, 0.01), loc=3, scale=2),
                facecolor='blue', alpha=0.35)

# H1 在拒絕域的部分 (Power)
ax.fill_between(x=np.arange(2, 10, 0.01),
                y1=stats.norm.pdf(np.arange(2, 10, 0.01), loc=3, scale=2),
                facecolor='grey', alpha=0.35)

# 中文標註
ax.text(x=-0.8, y=0.15, s='虛無假設 H$_0$', fontsize=13, fontweight='bold')
ax.text(x=3.0, y=0.14, s='對立假設 H$_1$', fontsize=13, fontweight='bold')
ax.text(x=2.1, y=0.012, s='Type I Error\n(偽陽性, $\\alpha$)',
        fontsize=10, color='darkred')
ax.text(x=-3.8, y=0.012, s='Type I Error\n(偽陽性, $\\alpha$)',
        fontsize=10, color='darkred')
ax.text(x=-0.2, y=0.025, s='Type II Error\n(偽陰性, $\\beta$)',
        fontsize=10, color='darkblue')

# 臨界值標線
ax.axvline(x=-2, color='black', linestyle='--', lw=1, alpha=0.5)
ax.axvline(x=2, color='black', linestyle='--', lw=1, alpha=0.5)
ax.text(x=-2, y=-0.02, s='$-z_{\\alpha/2}$', fontsize=11, ha='center')
ax.text(x=2, y=-0.02, s='$z_{\\alpha/2}$', fontsize=11, ha='center')

# 圖例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', alpha=0.4, label='Type I Error ($\\alpha$) — 偽陽性'),
    Patch(facecolor='blue', alpha=0.35, label='Type II Error ($\\beta$) — 偽陰性'),
    Patch(facecolor='grey', alpha=0.35, label='正確決策區域'),
]
ax.legend(handles=legend_elements, fontsize=11, loc='upper right')

ax.set_xlabel('檢定統計量', fontsize=12)
ax.set_ylabel('機率密度', fontsize=12)
ax.set_title('Type I Error 與 Type II Error 視覺化', fontsize=15)
ax.set_xlim(-4.5, 10)

plt.tight_layout()
plt.show()

In [ ]:
# 計算 Type II Error 機率
# H0: N(0,1), H1: N(3,2), alpha = 0.05 (雙尾)

lower_cutoff = stats.norm.ppf(0.025)   # 下臨界值
upper_cutoff = stats.norm.ppf(0.975)   # 上臨界值

# H1 分佈在不拒絕域（兩條臨界值之間）的面積 = Type II Error (beta)
beta = (stats.norm.cdf(upper_cutoff, loc=3, scale=2) -
        stats.norm.cdf(lower_cutoff, loc=3, scale=2))

power = 1 - beta

print(f'下臨界值: {lower_cutoff:.4f}')
print(f'上臨界值: {upper_cutoff:.4f}')
print(f'Type II Error (beta): {beta:.4f} ({beta*100:.1f}%)')
print(f'統計檢定力 (Power):    {power:.4f} ({power*100:.1f}%)')

### C2. Type I 與 Type II Error 的取捨

兩種錯誤之間存在**蹺蹺板效應**：

- 降低 $\alpha$（更嚴格的門檻）→ Type I Error 減少，但 Type II Error 增加
- 提高 $\alpha$（更寬鬆的門檻）→ Type I Error 增加，但 Type II Error 減少

在不同情境下，哪種錯誤的代價更大？

| 情境 | 更擔心哪種錯誤？ | 原因 |
|------|-----------------|------|
| 新藥上市審查 | Type I Error | 無效的藥被批准，危害病人 |
| 疾病篩檢 | Type II Error | 有病的人被漏診，延誤治療 |
| 刑事審判 | Type I Error | 「寧可放過，不可冤枉」 |

---
## Part D — 統計檢定力 (Statistical Power)

### D1. Power 的定義

$$
\text{Power} = 1 - \beta = P(\text{拒絕 } H_0 \mid H_0 \text{ 為假})
$$

**統計檢定力 (Power)** 就是：當 H$_0$ 確實為假時，檢定能**正確偵測到差異**的機率。

一般慣例：Power $\geq$ 0.8（80%）才算足夠。

### D2. 影響 Power 的三大因素

| 因素 | 對 Power 的影響 |
|------|----------------|
| **顯著水準 $\alpha$ 越大** | Power 越高（但 Type I Error 也越高） |
| **樣本大小 n 越大** | Power 越高（抽樣分佈越窄） |
| **效果量 (Effect Size) 越大** | Power 越高（兩分佈距離越遠） |

In [ ]:
# alpha 與 beta 的取捨：不同 alpha 下的錯誤率

alphas = [0.01, 0.05, 0.10, 0.20]
print(f"{"alpha":>8s} | {"Type I (alpha)":>16s} | {"Type II (beta)":>16s} | {"Power (1-beta)":>16s}")
print("-" * 65)

for a in alphas:
    # H1: N(3,2), 用雙尾臨界值
    z_lo = stats.norm.ppf(a / 2)
    z_hi = stats.norm.ppf(1 - a / 2)
    beta = stats.norm.cdf(z_hi, loc=3, scale=2) - stats.norm.cdf(z_lo, loc=3, scale=2)
    power = 1 - beta
    print(f"{a:>8.2f} | {a:>16.4f} | {beta:>16.4f} | {power:>16.4f}")

print("\n觀察: alpha 越大 -> Type I Error 越高，但 Power 越高（Type II Error 越低）")

In [ ]:
# Power Analysis：給定 alpha=0.05, power=0.8, effect_size=0.5
# 問：需要多少樣本？

n_required = tt_solve_power(effect_size=0.5,
                            alpha=0.05,
                            power=0.8)

print('=== 樣本數計算 (Power Analysis) ===')
print(f'效果量 (Effect Size): 0.5（中等）')
print(f'顯著水準 (alpha):     0.05')
print(f'目標檢定力 (Power):   0.80')
print(f'\n所需最小樣本數: {n_required:.1f} -> 至少 {int(np.ceil(n_required))} 個樣本')

In [ ]:
# 視覺化：樣本大小如何影響 Power

sample_sizes = np.arange(10, 200, 5)
powers = [tt_solve_power(effect_size=0.5, alpha=0.05, nobs=n)
          for n in sample_sizes]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sample_sizes, powers, 'b-', lw=2)
ax.axhline(y=0.8, color='red', linestyle='--', lw=1.5, label='Power = 0.80 門檻')
ax.axvline(x=int(np.ceil(n_required)), color='green', linestyle='--', lw=1.5,
           label=f'n = {int(np.ceil(n_required))}')

ax.set_xlabel('樣本大小 (n)', fontsize=12)
ax.set_ylabel('統計檢定力 (Power)', fontsize=12)
ax.set_title('樣本大小 vs 統計檢定力（Effect Size = 0.5, alpha = 0.05）', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 不同效果量下的 Power 曲線比較

fig, ax = plt.subplots(figsize=(10, 5))

effect_sizes = [0.2, 0.5, 0.8]
labels = ['小效果 (d=0.2)', '中效果 (d=0.5)', '大效果 (d=0.8)']
colors = ['#e74c3c', '#3498db', '#2ecc71']

for es, label, color in zip(effect_sizes, labels, colors):
    powers = [tt_solve_power(effect_size=es, alpha=0.05, nobs=n)
              for n in sample_sizes]
    ax.plot(sample_sizes, powers, lw=2, label=label, color=color)

ax.axhline(y=0.8, color='grey', linestyle='--', lw=1, alpha=0.7, label='Power = 0.80')
ax.set_xlabel('樣本大小 (n)', fontsize=12)
ax.set_ylabel('統計檢定力 (Power)', fontsize=12)
ax.set_title('不同效果量下的 Power 曲線', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part E — 效果量 (Effect Size)

In [ ]:
# 不同效果量所需的樣本數（alpha=0.05, power=0.80）

print("=== 單樣本 t 檢定：不同效果量所需的最小樣本數 ===")
print(f"{'Effect Size':>15s} | {'Cohen d':>10s} | {'所需樣本數':>10s}")
print("-" * 42)

for label, d in [("小效果", 0.2), ("中效果", 0.5), ("大效果", 0.8)]:
    n = tt_solve_power(effect_size=d, alpha=0.05, power=0.8)
    print(f"{label:>15s} | {d:>10.1f} | {int(np.ceil(n)):>10d}")

print("\n觀察: 效果量越小，需要越多樣本才能偵測到差異")

In [ ]:
# 雙樣本 t 檢定的 Power Analysis

n_two_sample = tt_ind_solve_power(effect_size=0.5,
                                  alpha=0.05,
                                  power=0.8,
                                  ratio=1.0)  # 兩組人數比例 1:1

print("=== 雙樣本檢定的樣本數計算 ===")
print(f"效果量: 0.5（中等）")
print(f"alpha: 0.05, Power: 0.80")
print(f"每組所需樣本數: {n_two_sample:.1f} -> 至少 {int(np.ceil(n_two_sample))} 人")
print(f"兩組合計至少: {int(np.ceil(n_two_sample)) * 2} 人")

### E1. Cohen's d

Cohen's d 是最常用的**標準化效果量**指標，衡量兩組均值差異相對於標準差的大小：

$$
d = \frac{\bar{X}_1 - \bar{X}_2}{s_{\text{pooled}}}
$$

其中 $s_{\text{pooled}} = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1 + n_2 - 2}}$

| Cohen's d | 效果大小 | 直覺解釋 |
|-----------|---------|----------|
| 0.2 | 小效果 (Small) | 需要仔細量測才能發現的差異 |
| 0.5 | 中效果 (Medium) | 肉眼可見的差異 |
| 0.8 | 大效果 (Large) | 明顯的差異 |

### E2. 為什麼 p-value 不夠？

p-value 受樣本大小的強烈影響：

- **大樣本** → 即使微小的差異也會產生極小的 p-value（統計顯著）
- **小樣本** → 即使實質差異很大，p-value 也可能不顯著

> 統計顯著 (Statistical Significance) $\neq$ 實務重要 (Practical Significance)

因此，報告結果時應同時呈現：
1. **p-value**：統計顯著性
2. **效果量 (Effect Size)**：實務重要性
3. **信賴區間 (Confidence Interval)**：估計的不確定性

In [ ]:
def cohens_d(group1, group2):
    """計算 Cohen's d 效果量"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    
    # Pooled standard deviation
    s_pooled = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    d = (group1.mean() - group2.mean()) / s_pooled
    return d


def interpret_cohens_d(d):
    """解釋 Cohen's d 的大小"""
    abs_d = abs(d)
    if abs_d < 0.2:
        return '可忽略 (Negligible)'
    elif abs_d < 0.5:
        return '小效果 (Small)'
    elif abs_d < 0.8:
        return '中效果 (Medium)'
    else:
        return '大效果 (Large)'

In [ ]:
# 實作：Titanic 男女年齡差異的 Cohen's d

male_age = titanic.loc[titanic['Sex'] == 'male', 'Age'].dropna()
female_age = titanic.loc[titanic['Sex'] == 'female', 'Age'].dropna()

d = cohens_d(male_age, female_age)

# 同時做 t 檢定
t_stat, p_val = stats.ttest_ind(male_age, female_age, equal_var=False)

print('=== Titanic 男女乘客年齡比較 ===')
print(f'男性: n={len(male_age)}, 平均={male_age.mean():.2f}, 標準差={male_age.std():.2f}')
print(f'女性: n={len(female_age)}, 平均={female_age.mean():.2f}, 標準差={female_age.std():.2f}')
print(f'\nt 統計量: {t_stat:.4f}')
print(f'p-value:  {p_val:.4f}')
print(f"Cohen's d: {d:.4f} → {interpret_cohens_d(d)}")
print(f'\n解讀: 雖然 p-value 可能顯示統計顯著差異，')
print(f'      但 Cohen\'s d = {d:.2f} 顯示效果量為「{interpret_cohens_d(d)}」，')
print(f'      在實務上差異並不大。')

In [ ]:
# 示範：大樣本讓微小差異也「統計顯著」

np.random.seed(42)
n = 100000  # 很大的樣本
group_a = np.random.normal(loc=100.0, scale=15, size=n)
group_b = np.random.normal(loc=100.3, scale=15, size=n)  # 只差 0.3

t_stat_big, p_val_big = stats.ttest_ind(group_a, group_b)
d_big = cohens_d(pd.Series(group_a), pd.Series(group_b))

print('=== 大樣本下的微小差異 ===')
print(f'n = {n:,} (每組)')
print(f'均值差: {group_a.mean():.3f} vs {group_b.mean():.3f}')
print(f'p-value: {p_val_big:.6f} → {"統計顯著" if p_val_big < 0.05 else "不顯著"}')
print(f"Cohen's d: {d_big:.4f} → {interpret_cohens_d(d_big)}")
print(f'\n結論: p-value 很小（統計顯著），但效果量幾乎為零 → 沒有實務意義！')

In [ ]:
# 視覺化不同 Cohen's d 的含義

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.linspace(-4, 6, 500)

for ax, d_val, title in zip(axes,
                            [0.2, 0.5, 0.8],
                            ["小效果 d=0.2", "中效果 d=0.5", "大效果 d=0.8"]):
    ax.plot(x, stats.norm.pdf(x, 0, 1), "b-", lw=2, label="Group 1")
    ax.plot(x, stats.norm.pdf(x, d_val, 1), "r-", lw=2, label="Group 2")
    ax.fill_between(x, stats.norm.pdf(x, 0, 1), alpha=0.15, color="blue")
    ax.fill_between(x, stats.norm.pdf(x, d_val, 1), alpha=0.15, color="red")
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 0.45)

plt.suptitle("Cohen's d：兩組分佈重疊程度", fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

---
## Part F — p-value 的常見誤解 (Common Misconceptions about p-value)

### F1. 五大迷思

| # | 常見誤解 | 正確理解 |
|---|---------|----------|
| 1 | p-value 是「H$_0$ 為真的機率」 | p-value 是「**假設 H$_0$ 為真時**，觀察到這麼極端結果的機率」 |
| 2 | p < 0.05 代表結果「重要」 | p < 0.05 只代表**統計顯著**，不代表**實務重要** |
| 3 | p > 0.05 代表「H$_0$ 為真」 | 只能說「沒有足夠證據拒絕 H$_0$」，**不能證明 H$_0$ 為真** |
| 4 | p-value 越小，效果越大 | p-value 受樣本大小影響，**小 p 不等於大效果** |
| 5 | 兩個研究的 p-value 可以直接比較 | p-value 不具可比性，應比較**效果量和信賴區間** |

### F2. 統計顯著 vs 實務重要

完整報告檢定結果應包含三個要素：

```
1. 統計顯著性: t(df) = 值, p = 值
2. 效果量:     Cohen's d = 值 [小/中/大]
3. 信賴區間:   95% CI [下界, 上界]
```

這樣才能同時回答：
- 差異是否**不太可能是偶然**的？（p-value）
- 差異**有多大**？（效果量）
- 估計的**不確定性**有多少？（信賴區間）

In [ ]:
# 完整報告範例：Titanic 存活者 vs 非存活者的年齡差異

survived_age = titanic.loc[titanic['Survived'] == 1, 'Age'].dropna()
died_age = titanic.loc[titanic['Survived'] == 0, 'Age'].dropna()

# t 檢定
t_stat, p_val = stats.ttest_ind(survived_age, died_age, equal_var=False)
df_approx = min(len(survived_age), len(died_age)) - 1  # 簡化的自由度

# 效果量
d = cohens_d(survived_age, died_age)

# 均值差的信賴區間
diff_mean = survived_age.mean() - died_age.mean()
se_diff = np.sqrt(survived_age.var(ddof=1)/len(survived_age) +
                  died_age.var(ddof=1)/len(died_age))
ci_low = diff_mean - 1.96 * se_diff
ci_high = diff_mean + 1.96 * se_diff

print('=' * 55)
print('完整檢定報告：Titanic 存活者 vs 罹難者的年齡差異')
print('=' * 55)
print(f'存活者: n={len(survived_age)}, M={survived_age.mean():.2f}, SD={survived_age.std():.2f}')
print(f'罹難者: n={len(died_age)}, M={died_age.mean():.2f}, SD={died_age.std():.2f}')
print(f'\n1. 統計顯著性: t({df_approx}) = {t_stat:.3f}, p = {p_val:.4f}')
print(f"2. 效果量:     Cohen's d = {d:.3f} [{interpret_cohens_d(d)}]")
print(f'3. 均值差 95% CI: [{ci_low:.2f}, {ci_high:.2f}]')
print(f'\n結論: ', end='')
if p_val < 0.05:
    print(f'存活者與罹難者年齡存在統計顯著差異 (p={p_val:.4f})，')
else:
    print(f'未發現統計顯著差異 (p={p_val:.4f})，')
print(f'效果量為{interpret_cohens_d(d)} (d={d:.3f})。')

---
## 重點整理

| 概念 | 關鍵要點 |
|------|----------|
| **假設檢定邏輯** | 設定 H$_0$ → 收集資料 → 計算 p-value → 與 $\alpha$ 比較 → 做出決策 |
| **p-value** | 在 H$_0$ 為真的前提下，觀察到這麼極端結果的機率 |
| **口訣** | 「p 小拒 H$_0$」 |
| **單尾 vs 雙尾** | 取決於研究假設是否有方向性 |
| **Type I Error** | 偽陽性（$\alpha$），H$_0$ 為真卻拒絕 |
| **Type II Error** | 偽陰性（$\beta$），H$_0$ 為假卻不拒絕 |
| **Power** | $1 - \beta$，受 $\alpha$、$n$、effect size 影響 |
| **Effect Size** | 量化實際差異大小，補足 p-value 的不足 |
| **完整報告** | p-value + effect size + 信賴區間 |

---
## 練習題

### 基礎題

1. **假設建立：** 研究者想知道 Titanic 一等艙乘客的平均票價是否超過 60 元。請寫出 H$_0$ 和 H$_1$，並說明這是單尾還是雙尾檢定。

2. **決策判斷：** 一個檢定結果 p-value = 0.03。
   - 若 $\alpha$ = 0.05，你的決策是什麼？
   - 若 $\alpha$ = 0.01，你的決策是什麼？

3. **錯誤辨別：** 法院審判中，H$_0$ 是「被告無罪」。請問判決一個無辜的人有罪是哪種錯誤？放走一個有罪的人呢？

### 進階題

4. **Power Analysis：** 你計畫研究一種新教學法是否比傳統教學法有效。預期效果量為 Cohen's d = 0.3（小效果），希望 power = 0.9，$\alpha$ = 0.05。使用 `tt_ind_solve_power` 計算每組需要多少人。

5. **Effect Size 計算：** 計算 Titanic 中一等艙 vs 三等艙乘客年齡的 Cohen's d，並解釋其實務意義。

### 挑戰題

6. **p-value 迷思破解：** 你的同事說「p = 0.001 代表 H$_0$ 只有 0.1% 的機率為真」。請用 Part F 的知識解釋這句話哪裡有問題，並給出正確的解讀。

### 思考題

> 一間製藥公司進行新藥臨床試驗，招募了 50,000 名受試者。
> 結果顯示 p = 0.001，但新藥只比安慰劑多降低了 0.1 mmHg 的血壓。
>
> 請問：
> 1. 這個結果「統計顯著」嗎？
> 2. 這個結果「有實務價值」嗎？
> 3. 你會建議這間公司推出這款新藥嗎？為什麼？

---

$\leftarrow$ [Ch06 — 點估計與信賴區間](06_點估計與信賴區間.ipynb) | [Ch08 — t 檢定與 z 檢定 $\rightarrow$](08_t檢定與z檢定.ipynb)

### F4. 實務決策指引

當你拿到一個 p-value 時，問自己這些問題：

1. **p < $\alpha$ 嗎？** → 判斷統計顯著性
2. **效果量有多大？** → 判斷實務重要性
3. **樣本大小合理嗎？** → 檢查 Power 是否足夠
4. **信賴區間包含什麼範圍？** → 評估估計的精確度
5. **結果可重複嗎？** → 單一研究的 p-value 不是最終答案

### F3. 生活化類比：法庭審判

假設檢定與法庭審判有驚人的相似之處：

| 假設檢定 | 法庭審判 |
|----------|---------|
| H$：無差異 | 被告無罪（無罪推定原則） |
| H$：有差異 | 被告有罪 |
| 收集資料 | 蒐集證據 |
| p-value 很小 | 證據強烈指向有罪 |
| 拒絕 H$ | 判決有罪 |
| 不拒絕 H$ | 證據不足，判決無罪（不代表確實無辜） |
| Type I Error | 冤枉好人 |
| Type II Error | 放走壞人 |
| $\alpha$ = 顯著水準 | 定罪的證據門檻 |